In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical Events - Dx (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - Dx (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical Events - NDC codes (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Medical Events - Procedure codes (RENDERING_NPI only)
SELECT DISTINCT 
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                         'S9357', 'S9379', '38206', '38230', '38232', 
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 3: TREATMENT CLAIMS FOR ELIGIBILITY (2yr)
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
-- =============================================================================

-- Specified: 2+ E761 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Specified Patients: 2+ E761 Dx + any Tx in 2yr
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;


-- Incremental: 2+ E763 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Elaprase Tx in 2yr (for incremental eligibility)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');


-- Incremental Patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);


-- All Eligible Patients
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- Dx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

-- Tx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
-- =====================================================
-- Create non-unique patient counts at HCP level
-- Includes ALL eligible patients (Dx + Tx)
-- Patients may be counted across multiple HCPs
-- =====================================================

CREATE OR REPLACE TEMPORARY VIEW all_patients_non_unique_hcp AS

WITH hcp_level AS (
    SELECT *
    FROM all_patient_claims
)

SELECT
    TRY_CAST(npi AS STRING) AS hcp_npi,
    COUNT(DISTINCT patient_id) AS all_patients_non_unique
FROM hcp_level
WHERE npi IS NOT NULL
GROUP BY 1
ORDER BY 2 DESC;


In [0]:
-- ============================================================
-- Create UNIQUE eligible patient counts at HCP level
-- Each patient is assigned to ONE primary HCP
-- Attribution logic uses combined Dx + Tx activity
-- ============================================================

CREATE OR REPLACE TEMPORARY VIEW all_patients_unique_hcp AS

WITH hcp_metrics AS (
    SELECT 
        a.PATIENT_ID,
        a.NPI,

        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 'Geneticist'

            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 'Psychiatry & Neurology'

            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 'Pediatrician'

            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 'PCP'

            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 'NPPA'

            WHEN a.NPI IS NULL 
                THEN 'NA'

            ELSE 'Others'
        END AS SPECIALTY,

        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 5
            WHEN a.NPI IS NULL 
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        COUNT(DISTINCT CASE 
            WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE 
        END) AS DX_VISITS,

        COUNT(DISTINCT CASE 
            WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE 
        END) AS TX_VISITS,

        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p 
        ON a.NPI = p.NPI

    GROUP BY 
        a.PATIENT_ID, 
        a.NPI,
        p.primary_specialty, 
        p.secondary_specialty
),

ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
),

hcp_level AS (
    SELECT 
        PATIENT_ID,
        NPI AS PRIMARY_HCP_NPI,
        SPECIALTY AS PRIMARY_HCP_SPECIALTY,
        SPECIALTY_PRIORITY,
        NO_OF_VISITS,
        DX_VISITS,
        TX_VISITS,
        MOST_RECENT_VISIT
    FROM ranked_hcps
    WHERE HCP_RANK = 1
)

-- ============================================================
-- Final UNIQUE patient count by HCP
-- Each patient contributes to exactly ONE primary HCP
-- ============================================================
SELECT
    TRY_CAST(primary_hcp_npi AS STRING) AS hcp_npi,
    primary_hcp_specialty,
    COUNT(DISTINCT patient_id) AS all_patients_unique
FROM hcp_level
WHERE primary_hcp_npi IS NOT NULL
GROUP BY 1, 2
ORDER BY 3 DESC;


In [0]:
WITH input_hcps AS (
  SELECT col1 AS hcp_npi
  FROM VALUES
  ('1457745341'),('1245766757'),('1407132665'),('1659459444'),('1083841191'),
  ('1457439713'),('1225113954'),('1295946176'),('1306284898'),('1730541491'),
  ('1699798603'),('1326562992'),('1770900904'),('1154521722'),('1356798540'),
  ('1073994448'),('1922354166'),('1124405311'),('1396424826'),('1104906445'),
  ('1861866717'),('1376836783'),('1225365307'),('1467598508'),('1679893010'),
  ('1710145362'),('1790216083'),('1083960884'),('1497118004'),('1952036337'),
  ('1295291938'),('1831628346'),('1568151975'),('1689243842'),('1538514914'),
  ('1184373508'),('1104598333'),('1730969783'),('1700812500'),('1336260280'),
  ('1568548246'),('1558560169'),('1417311788'),('1417593237'),('1619646676'),
  ('1457127995'),('1760916860'),('1538109558'),('1053539015'),('1275051534'),
  ('1548604895'),('1891995122'),('1154515963'),('1770296790'),('1003522897'),
  ('1760561336'),('1558820373'),('1922386119'),('1255435301'),('1033539671'),
  ('1720127194'),('1679617930'),('1902009418'),('1508914458'),('1104395656'),
  ('1386481000'),('1801316526'),('1447918628'),('1699880856'),('1780878132'),
  ('1114294477'),('1508961897'),('1205292133'),('1932603289'),('1497045298'),
  ('1912964925'),('1275612749'),('1134199946'),('1831692375'),('1497071963'),
  ('1710064357'),('1427136175'),('1811075567'),('1093249872'),('1013179332'),
  ('1679686653'),('1275796419'),('1972868073'),('1922241686'),('1851785067'),
  ('1295211985'),('1639804339'),('1780931956'),('1902012180'),('1750720652'),
  ('1992766695'),('1023647062'),('1255346771'),('1326306721'),('1013726561'),
  ('1699743088'),('1134534597'),('1467848366'),('1720507486'),('1730682576'),
  ('1942755368'),('1821665985'),('1285610311'),('1912071937'),('1679500292'),
  ('1407284367'),('1437553906'),('1609003011'),('1861623985'),('1932429917'),
  ('1588735005'),('1851536072'),('1700173895'),('1194056630'),('1720378904'),
  ('1780888644'),('1821479957'),('1801812128'),('1912087354'),('1740218270'),
  ('1104486638'),('1689651218'),('1033387105'),('1528164159'),('1811934490'),
  ('1538373816'),('1780635839'),('1003203779'),('1336651918'),('1083197727'),
  ('1255824280'),('1497723662'),('1285021774'),('1811263627'),('1649478553'),
  ('1063032688'),('1477736163'),('1508401076'),('1689037434'),('1740961929'),
  ('1518629294'),('1376815712'),('1467011197'),('1679712087'),('1013229426'),
  ('1649488164'),('1376521278'),('1427047380'),('1629686977'),('1346288461'),
  ('1194067470'),('1851010797'),('1740296946'),('1912511452'),('1366935439'),
  ('1659563088'),('1508540626'),('1194986554'),('1760877088'),('1578811469'),
  ('1356372783'),('1467433946'),('1528223682'),('1083233993'),('1497958243'),
  ('1417117243'),('1538133194'),('1699138685'),('1669821781'),('1154431567'),
  ('1225476930'),('1740566561'),('1942760681'),('1316137060'),('1346656774'),
  ('1861667065'),('1144498429'),('1831455690'),('1053840074'),('1053655514'),
  ('1487958146'),('1700044336'),('1023492980'),('1144715798'),('1700447216'),
  ('1992122212'),('1053393470'),('1982799417'),('1326475633'),('1841271806'),
  ('1376527861'),('1003850058'),('1386686467'),('1730239153'),('1699157982'),
  ('1215983382'),('1285623074'),('1972506749'),('1093774804'),('1740292903'),
  ('1568728806'),('1851403539'),('1073709457'),('1356628887'),('1770759029'),
  ('1043662299'),('1023066198'),('1356429518'),('1194808535'),('1063877058'),
  ('1972765220'),('1629050851'),('1366671380'),('1114377199'),('1548362759'),
  ('1740202159'),('1790313666'),('1013293380'),('1477999522'),('1164957437'),
  ('1861430175'),('1710644190'),('1649211798'),('1457601692'),('1225478233'),
  ('1730421611'),('1295177483'),('1407876584'),('1619979697'),('1831164672'),
  ('1861662975'),('1033345509'),('1548266257'),('1114949617'),('1205991361'),
  ('1194823963'),('1770877904'),('1912533555'),('1114585056'),('1245049667'),
  ('1326022278'),('1811277130'),('1447132543'),('1295132165'),('1053193029'),
  ('1912973447'),('1215118625'),('1164952099'),('1043970064'),('1497893846'),
  ('1639124530'),('1417000357'),('1114269479'),('1114439882'),('1780368944'),
  ('1427270313'),('1164733960'),('1255106431'),('1790716595'),('1255381794'),
  ('1831403211'),('1760183412'),('1811579022'),('1134149495'),('1346668399'),
  ('1992784532'),('1144626284'),('1154883148'),('1770872350'),('1982138483'),
  ('1699983155'),('1881076248'),('1568454981'),('1881651396'),('1790171825'),
  ('1023452885'),('1568624633'),('1609635499'),('1467474502'),('1740771617'),
  ('1164829768'),('1215988977'),('1578520250'),('1144703638'),('1275841603'),
  ('1417210170'),('1386671394'),('1205896933'),('1104010982'),('1891354510'),
  ('1093078305'),('1326420662'),('1801987664'),('1962923862'),('1720528011'),
  ('1043768005'),('1366829152'),('1942612049'),('1578129607'),('1114001021'),
  ('1306926944'),('1659904811'),('1942545314'),('1750115127'),('1538184940'),
  ('1922569649'),('1982996435'),('1346801164'),('1972554897'),('1588984561'),
  ('1306076039'),('1407194038'),('1861897241'),('1427599034'),('1891071478'),
  ('1659717148'),('1215199146'),('1326122441'),('1437793080'),('1013109594')
),

-- Enrich from reference file but KEEP all input NPIs
base_table AS (
  SELECT
    i.hcp_npi,
    COALESCE(
      r.hcp_name,
      NULLIF(TRIM(CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME)), '')
    ) AS hcp_name,
    b.PRIMARY_SPECIALTY,
    b.SECONDARY_SPECIALTY,
    r.hco_npi,
    r.hco_name
  FROM input_hcps i
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 r
    ON TRY_CAST(i.hcp_npi AS STRING) = TRY_CAST(r.hcp_npi AS STRING)
  LEFT JOIN com_edp_prd.com_raw.kom_providers b
    ON TRY_CAST(b.npi AS STRING) = TRY_CAST(i.hcp_npi AS STRING)
    AND UPPER(b.PROVIDER_TYPE) = 'INDIVIDUAL'
),


hcp_engaged AS (
  SELECT DISTINCT
    TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_edp_prd.com_raw.vcrm_call2__v AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  INNER JOIN input_hcps i
    ON TRY_CAST(b.npi__v AS STRING) = TRY_CAST(i.hcp_npi AS STRING)
  WHERE a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hcp_profiled AS (
  SELECT DISTINCT
    TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_intgr.survey_target AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  INNER JOIN input_hcps i
    ON TRY_CAST(b.npi__v AS STRING) = TRY_CAST(i.hcp_npi AS STRING)
)

SELECT DISTINCT
  a.hcp_npi,
  a.hcp_name,
  a.primary_specialty,
  a.secondary_specialty,
  a.hco_npi,
  a.hco_name,

  CASE WHEN d.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hcp_engaged,
  CASE WHEN e.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hcp_profiled,

  COALESCE(nu.all_patients_non_unique, 0) AS all_patients_non_unique,
  COALESCE(u.all_patients_unique, 0)      AS all_patients_unique

FROM base_table a
LEFT JOIN hcp_engaged d
  ON TRY_CAST(a.hcp_npi AS STRING) = d.npi
LEFT JOIN hcp_profiled e
  ON TRY_CAST(a.hcp_npi AS STRING) = e.npi

LEFT JOIN all_patients_non_unique_hcp nu
  ON TRY_CAST(a.hcp_npi AS STRING) = TRY_CAST(nu.hcp_npi AS STRING)

LEFT JOIN (
  -- ensure 1 row per HCP in case your unique view has multiple rows per HCP
  SELECT
    TRY_CAST(hcp_npi AS STRING) AS hcp_npi,
    MAX(all_patients_unique) AS all_patients_unique
  FROM all_patients_unique_hcp
  GROUP BY 1
) u
  ON TRY_CAST(a.hcp_npi AS STRING) = u.hcp_npi
;


In [0]:
WITH input_hcps AS (
  SELECT col1 AS hcp_npi
  FROM VALUES
  ('1457745341'),('1245766757'),('1407132665'),('1659459444'),('1083841191'),
  ('1457439713'),('1225113954'),('1295946176'),('1306284898'),('1730541491'),
  ('1699798603'),('1326562992'),('1770900904'),('1154521722'),('1356798540'),
  ('1073994448'),('1922354166'),('1124405311'),('1396424826'),('1104906445'),
  ('1861866717'),('1376836783'),('1225365307'),('1467598508'),('1679893010'),
  ('1710145362'),('1790216083'),('1083960884'),('1497118004'),('1952036337'),
  ('1295291938'),('1831628346'),('1568151975'),('1689243842'),('1538514914'),
  ('1184373508'),('1104598333'),('1730969783'),('1700812500'),('1336260280'),
  ('1568548246'),('1558560169'),('1417311788'),('1417593237'),('1619646676'),
  ('1457127995'),('1760916860'),('1538109558'),('1053539015'),('1275051534'),
  ('1548604895'),('1891995122'),('1154515963'),('1770296790'),('1003522897'),
  ('1760561336'),('1558820373'),('1922386119'),('1255435301'),('1033539671'),
  ('1720127194'),('1679617930'),('1902009418'),('1508914458'),('1104395656'),
  ('1386481000'),('1801316526'),('1447918628'),('1699880856'),('1780878132'),
  ('1114294477'),('1508961897'),('1205292133'),('1932603289'),('1497045298'),
  ('1912964925'),('1275612749'),('1134199946'),('1831692375'),('1497071963'),
  ('1710064357'),('1427136175'),('1811075567'),('1093249872'),('1013179332'),
  ('1679686653'),('1275796419'),('1972868073'),('1922241686'),('1851785067'),
  ('1295211985'),('1639804339'),('1780931956'),('1902012180'),('1750720652'),
  ('1992766695'),('1023647062'),('1255346771'),('1326306721'),('1013726561'),
  ('1699743088'),('1134534597'),('1467848366'),('1720507486'),('1730682576'),
  ('1942755368'),('1821665985'),('1285610311'),('1912071937'),('1679500292'),
  ('1407284367'),('1437553906'),('1609003011'),('1861623985'),('1932429917'),
  ('1588735005'),('1851536072'),('1700173895'),('1194056630'),('1720378904'),
  ('1780888644'),('1821479957'),('1801812128'),('1912087354'),('1740218270'),
  ('1104486638'),('1689651218'),('1033387105'),('1528164159'),('1811934490'),
  ('1538373816'),('1780635839'),('1003203779'),('1336651918'),('1083197727'),
  ('1255824280'),('1497723662'),('1285021774'),('1811263627'),('1649478553'),
  ('1063032688'),('1477736163'),('1508401076'),('1689037434'),('1740961929'),
  ('1518629294'),('1376815712'),('1467011197'),('1679712087'),('1013229426'),
  ('1649488164'),('1376521278'),('1427047380'),('1629686977'),('1346288461'),
  ('1194067470'),('1851010797'),('1740296946'),('1912511452'),('1366935439'),
  ('1659563088'),('1508540626'),('1194986554'),('1760877088'),('1578811469'),
  ('1356372783'),('1467433946'),('1528223682'),('1083233993'),('1497958243'),
  ('1417117243'),('1538133194'),('1699138685'),('1669821781'),('1154431567'),
  ('1225476930'),('1740566561'),('1942760681'),('1316137060'),('1346656774'),
  ('1861667065'),('1144498429'),('1831455690'),('1053840074'),('1053655514'),
  ('1487958146'),('1700044336'),('1023492980'),('1144715798'),('1700447216'),
  ('1992122212'),('1053393470'),('1982799417'),('1326475633'),('1841271806'),
  ('1376527861'),('1003850058'),('1386686467'),('1730239153'),('1699157982'),
  ('1215983382'),('1285623074'),('1972506749'),('1093774804'),('1740292903'),
  ('1568728806'),('1851403539'),('1073709457'),('1356628887'),('1770759029'),
  ('1043662299'),('1023066198'),('1356429518'),('1194808535'),('1063877058'),
  ('1972765220'),('1629050851'),('1366671380'),('1114377199'),('1548362759'),
  ('1740202159'),('1790313666'),('1013293380'),('1477999522'),('1164957437'),
  ('1861430175'),('1710644190'),('1649211798'),('1457601692'),('1225478233'),
  ('1730421611'),('1295177483'),('1407876584'),('1619979697'),('1831164672'),
  ('1861662975'),('1033345509'),('1548266257'),('1114949617'),('1205991361'),
  ('1194823963'),('1770877904'),('1912533555'),('1114585056'),('1245049667'),
  ('1326022278'),('1811277130'),('1447132543'),('1295132165'),('1053193029'),
  ('1912973447'),('1215118625'),('1164952099'),('1043970064'),('1497893846'),
  ('1639124530'),('1417000357'),('1114269479'),('1114439882'),('1780368944'),
  ('1427270313'),('1164733960'),('1255106431'),('1790716595'),('1255381794'),
  ('1831403211'),('1760183412'),('1811579022'),('1134149495'),('1346668399'),
  ('1992784532'),('1144626284'),('1154883148'),('1770872350'),('1982138483'),
  ('1699983155'),('1881076248'),('1568454981'),('1881651396'),('1790171825'),
  ('1023452885'),('1568624633'),('1609635499'),('1467474502'),('1740771617'),
  ('1164829768'),('1215988977'),('1578520250'),('1144703638'),('1275841603'),
  ('1417210170'),('1386671394'),('1205896933'),('1104010982'),('1891354510'),
  ('1093078305'),('1326420662'),('1801987664'),('1962923862'),('1720528011'),
  ('1043768005'),('1366829152'),('1942612049'),('1578129607'),('1114001021'),
  ('1306926944'),('1659904811'),('1942545314'),('1750115127'),('1538184940'),
  ('1922569649'),('1982996435'),('1346801164'),('1972554897'),('1588984561'),
  ('1306076039'),('1407194038'),('1861897241'),('1427599034'),('1891071478'),
  ('1659717148'),('1215199146'),('1326122441'),('1437793080'),('1013109594')
)
select distinct npi
from com_raw.kom_providers
where PROVIDER_TYPE = 'INDIVIDUAL'
and npi in (select distinct hcp_npi from input_hcps)
--  group by 1 order by 2 desc